# TaaS Agent & MCP Registry Studio — Colab validation

**Target PR:** [#27 — Versioned MCP snapshots and Agent tool-grant binding](https://github.com/msellamiTN/taas-provenance-discovery/pull/34)

This notebook is a fresh operator validation entry point pinned to one immutable commit. It launches the real Studio API in this Colab runtime and provides basic health, readiness, authenticated Agent Registry and MCP snapshot API smoke checks.

**Scope:** this validates launch and API reachability. It does not claim live MCP server inspection unless an operator-configured MCP server is available; it does not invoke MCP tools or enable governed execution. GPU availability is diagnostic only.

In [ ]:
import os, sys, subprocess, pathlib, time, urllib.request, urllib.error, json, secrets
from IPython.display import display, HTML

REPO_URL = "https://github.com/msellamiTN/taas-provenance-discovery.git"
TAAS_COMMIT = "729a1b6922deba7f2c57a96a899514b787e3c7e6"
WORKDIR = pathlib.Path("/content/taas-provenance-discovery")
APP_BIND_HOST, APP_HEALTH_HOST, APP_PORT = "0.0.0.0", "127.0.0.1", 8000

if not WORKDIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(WORKDIR)], check=True)
subprocess.run(["git", "fetch", "--all"], cwd=WORKDIR, check=True)
subprocess.run(["git", "checkout", "--detach", TAAS_COMMIT], cwd=WORKDIR, check=True)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=WORKDIR, text=True).strip()
assert actual == TAAS_COMMIT, (actual, TAAS_COMMIT)
assert (WORKDIR / "pyproject.toml").is_file()
print("Pinned TaaS revision:", actual)

## Runtime diagnostics

The discovery and registry APIs are CPU-oriented. A GPU is only relevant if you separately load a local model in Model Studio; this notebook does not infer model compatibility from the GPU probe.

In [ ]:
gpu = subprocess.run(["bash", "-lc", "nvidia-smi"], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else "No NVIDIA GPU allocated; continuing in CPU mode.")
print("Python:", sys.version.split()[0])

## Install dependencies

Installs the exact checked-out revision with the Studio and Hugging Face extras. Import failure stops the notebook and prints the diagnostic tail.

In [ ]:
install = subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[studio,hf]"], cwd=WORKDIR, text=True, capture_output=True)
print(install.stdout[-5000:])
if install.returncode:
    print(install.stderr[-5000:])
    raise RuntimeError("TaaS installation failed")
check = subprocess.run([sys.executable, "-c", "import taas, fastapi, uvicorn, torch, transformers, safetensors; import taas.studio.api; print('Studio imports OK')"], cwd=WORKDIR, text=True, capture_output=True)
print(check.stdout)
if check.returncode:
    print(check.stderr[-12000:])
    raise RuntimeError("TaaS import check failed")

## Launch the API

The API binds to `0.0.0.0` for Colab port proxy access. Health/readiness probes remain loopback-only. The bearer token is generated for this runtime and must not be shared or committed.

In [ ]:
api_token = secrets.token_urlsafe(32)
os.environ["TAAS_STUDIO_API_TOKEN"] = api_token
os.environ.pop("TAAS_MODEL_PROFILE", None)
print("Temporary Studio API token — copy into the Studio header; do not share:")
print(api_token)

if "studio_process" in globals() and studio_process.poll() is None:
    studio_process.terminate()
    try: studio_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        studio_process.kill(); studio_process.wait(timeout=5)
if "studio_log" in globals() and not studio_log.closed:
    studio_log.close()
studio_log = open("/content/taas-studio-pr34.log", "a", buffering=1)
studio_process = subprocess.Popen([sys.executable, "-m", "uvicorn", "taas.studio.api:app", "--host", APP_BIND_HOST, "--port", str(APP_PORT)], cwd=WORKDIR, env=os.environ.copy(), stdout=studio_log, stderr=subprocess.STDOUT)

def get_local(path, headers=None, attempts=45):
    url = f"http://{APP_HEALTH_HOST}:{APP_PORT}{path}"
    last = None
    for _ in range(attempts):
        if studio_process.poll() is not None:
            studio_log.flush()
            raise RuntimeError("Studio exited; inspect /content/taas-studio-pr34.log")
        try:
            req = urllib.request.Request(url, headers=headers or {})
            with urllib.request.urlopen(req, timeout=4) as res:
                return res.status, res.read().decode("utf-8")
        except urllib.error.HTTPError as exc:
            return exc.code, exc.read().decode("utf-8")
        except Exception as exc:
            last = exc; time.sleep(1)
    raise RuntimeError(f"Studio did not become ready: {last}")

print("Health:", get_local("/health"))
print("Readiness:", get_local("/ready"))
print("PID:", studio_process.pid)

## Authenticated registry smoke tests

These checks verify that the authenticated Agent Registry and MCP snapshot list endpoints are reachable. An empty list is a valid result when no Agent versions or MCP snapshots have been created. This does not perform live MCP inspection.

In [ ]:
auth = {"Authorization": f"Bearer {api_token}"}
for path in ["/api/v1/agents", "/api/v1/mcp/registry/snapshots"]:
    status, body = get_local(path, headers=auth, attempts=3)
    print(f"{path}: HTTP {status}")
    print(body[:3000])
    assert status == 200, f"Unexpected response for {path}: {status}"

## Open Studio in a new browser tab

Keep the Colab runtime connected while using the Studio. Paste the temporary token into the Studio authentication field.

In [ ]:
from google.colab import output
proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({APP_PORT})")
display(HTML(f'<p><a href="{proxy_url}" target="_blank" rel="noopener">Open TaaS Agent & MCP Registry Studio</a></p>'))

## Operator acceptance checklist

1. Confirm the pinned commit printed above equals `729a1b6922deba7f2c57a96a899514b787e3c7e6`.
2. Confirm installation and imports succeed.
3. Confirm `/health` and `/ready` return HTTP 200.
4. Confirm both authenticated registry list endpoints return HTTP 200.
5. Open Studio, connect using the temporary token, and inspect Agent Studio and Tools & MCP.
6. If a test MCP server is configured by the operator, run the read-only inspection workflow and verify a stored immutable snapshot. Do not use production secrets or production MCP servers for this test.
7. Record Colab runtime type, Python version, GPU allocation, outputs and any traceback against this exact commit.

**Not covered:** actual MCP `tools/call`, governed execution, isolation, secrets broker, durable traces, multi-tenant operation or production readiness.

In [ ]:
if "studio_process" in globals() and studio_process.poll() is None:
    studio_process.terminate()
    try: studio_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        studio_process.kill(); studio_process.wait(timeout=5)
if "studio_log" in globals() and not studio_log.closed:
    studio_log.flush(); studio_log.close()
print("TaaS Studio stopped.")